# TopoMT DFND Delaunay Tetrahedra Visualization Smoke Test

This notebook demonstrates loading a synthetic shape benchmark structure, calculating its topography using the **DFND** (Delaunay Flow Network pocket Detector) algorithm, and visualizing both the molecular pockets (as density spheres) and the Delaunay tetrahedra color-coded by their physical classification (wet, dry, coast, etc.).

We will showcase both the **programmatic API** and the interactive **GUI panels** in `molsysviewer`.

In [1]:
import os
import numpy as np
import molsysmt as msm
import topomt as tmt
import molsysviewer_topomt as msv_tmt
import pyunitwizard as puw

## 1. Load a Synthetic PDB Benchmark

Let's load a synthetic PDB benchmark structure. These synthetic shapes are stored in `topomt/data/synthetic`.

In [2]:
# molecular_system = tmt.dfnd.synthetic.argon_cube(probe_radius=1.4)

In [3]:
# Locate the hollow_sphere_pocket.pdb benchmark file
pdb_path = tmt.demo['synthetic']['argon_cube.pdb']

# Convert to a molecular system
molecular_system = msm.convert(pdb_path)
print(f"Successfully loaded. Number of atoms: {msm.get(molecular_system, element='system', n_atoms=True)}")

Successfully loaded. Number of atoms: 8


In [4]:
msm.info(molecular_system)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_small_molecules,n_structures
molsysmt.MolSys,8,8,8,1,8,8,8,1


In [ ]:
#%%time
#msm.view(system)

## 2. Calculate Topography with DFND Method

Now we run the **Delaunay Flow Network pocket Detector (DFND)** algorithm on the molecular system. DFND partitions the Delaunay triangulation into topological and pocket-related domains.

In [5]:
# Run the DFND algorithm
print("Calculating topography using DFND...")
topography = tmt.get_topography(molecular_system, method='dfnd', probe_radius=1.4)
print("Topography calculation complete!")

Calculating topography using DFND...
Topography calculation complete!


In [6]:
print("=" * 50)                                                                                                                                                       
print("✅ Cubo de Argón inicializado exitosamente.")                                                                                                                  
print(f" • Número de átomos: {msm.get(molecular_system, n_atoms=True)}")                                                                                              
print(f" • Tetraedros de Delaunay totales: {len(topography.dfnd.raw['tetrahedra'])}")                                                                                 
print("=" * 50) 

✅ Cubo de Argón inicializado exitosamente.
 • Número de átomos: 8
 • Tetraedros de Delaunay totales: 6


In [7]:
# Check found features
pockets = topography.get_features(by='type', value='pocket')
voids = topography.get_features(by='type', value='void')
channels = topography.get_features(by='type', value='channel')

print(f"Found pockets: {len(pockets)}")
print(f"Found voids: {len(voids)}")
print(f"Found channels: {len(channels)}")

Found pockets: 0
Found voids: 1
Found channels: 0


In [8]:
#topography.dfnd.mesh.tetrahedra

## 3. Visualizing pockets and Delaunay Tetrahedra

We can build an interactive viewer using `molsysviewer_topomt.new_view`. Since the topography already has a reference to the molecular system internally, we only need to pass the topography as the main argument. By setting `render_tetrahedra=True`, the color-coded Delaunay tetrahedra are displayed immediately alongside the pockets.

### GUI Interaction:
- Check the **TopoMT Topography** panel in the right sidebar.
- You will see a premium violet button: **Render Tetrahedra**.
- Clicking **Clear** will cleanly wipe both pocket blobs and the tetrahedra layer!
- Clicking **Render pockets** and **Render Tetrahedra** lets you toggle them interactively.

In [9]:
# Build the interactive view and render both pockets and tetrahedra
view = msv_tmt.new_view(
    topography,
    render_tetrahedra=True,
    color_mode='combined_class',
    alpha=0.45,                                                                                                                                                       
    draw_edges=True,                                                                                                                                                  
    edge_radius_nm=0.003, # Bordes finos de soporte                                                                                                                   
    edge_color=0x333333       
)

In [10]:
view

In [13]:
msv_tmt.attach_dfnd_tetrahedra(
        view, 
        topography,
        color_mode='combined_class',
        alpha=0.55,
        draw_edges=True,
        edge_radius_nm=0.003,
        edge_color=0x222222,
        show_all_faces=True # <-- ¡Dibuja aristas y caras internas completas!
    )

{'addon_enabled': True,
 'layer': <molsysviewer.layers.Shape at 0x714f07141a70>,
 'tag': 'dfnd-tetra'}

## 4. Programmatic API Control

Every GUI action has a direct, reproducible programmatic API equivalent. Let's try toggling the representation programmatically.

In [12]:
# Programmatically render tetrahedra with different color modes and parameters
# Color modes can be:
# - 'combined_class' (wet_sealed, wet_mouth, coast, dry_contact, dry_motif, dry_bank)
# - 'transit_role' (resident_transit, gate_transit, none)
# - 'residence_state' (resident, boundary, none)

msv_tmt.render_dfnd_tetrahedra(
    view,
    topography,
    color_mode='transit_role',
    alpha=0.3,
    draw_edges=True,
    edge_radius_nm=0.008
)

In [18]:
topography.dfnd.raw

{'parameters': {'probe_radius': 1.4,
  'epsilon_length': 1e-06,
  'sea_level': None,
  'radii_model': 'vdw',
  'selection': 'all',
  'hydrogen_policy': 'exclude',
  'transit_policy': 'with_connectors',
  'gate_intrusion_policy': 'flag_only'},
 'tetrahedra': [{'tetrahedron_id': 0,
   'atom_indices': [0, 2, 3, 4],
   'local_atom_indices': [0, 2, 3, 4],
   'R_residence': 1.4005042295354535,
   'residence_candidate_kind': 'interior4',
   'R_apollonius4': 1.4005042295354535,
   'apollonius4_valid': True,
   'residence_margin': 0.0005042295354535931,
   'center': [0.0, -4.440892098500626e-16, 4.440892098500626e-16],
   'volume_topological': 9.058966645333332,
   'volume_solvent_estimate': 4.2275177678222216,
   'solvent_empty_fraction_estimate': 0.4666666666666667,
   'solvent_occupied_fraction_estimate': 0.5333333333333333,
   'solvent_volume_n_samples': 165,
   'residence_state': 'resident',
   'transit_role': 'resident_transit',
   'n_permeable_contacts': 2,
   'local_class': 'coast',
   